# EXP3 Test Notebook (Tagged Nico Sentences)

This notebook runs **EXP3** (XLM-R + style + char + POS + lemma) on the tagged Nico sentences Excel and saves predictions.

## Run order (after Kernel Reset)
1. **Cell 1** – Config: set `BASE_DIR`, input Excel, and paths to `exp3_model_state.pt` and `style_scaler_EXP1.pkl`
2. **Cell 2** – Imports + device
3. **Cell 3** – Load Excel
4. **Cell 4** – Style/char/POS/lemma feature builders
5. **Cell 5** – Load EXP3 model
6. **Cell 6** – Inference + save Excel

If you get an error about missing `pos/lemma` columns, you need to generate/tag them for this Excel (or merge from your cached tagging output) before running inference.


In [1]:
# ============================
# 1) Config (EDIT ME)
# ============================
import os

# Project root on your machine (EDIT)
BASE_DIR = r"C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית"  # <-- EDIT if needed

# Input Excel
INPUT_XLSX = os.path.join(
    BASE_DIR, "data", "inputs",
    r"Tagged_Nico_TAG 5 vs Tag 4 (update model 6+5) vs TAG 0  (antiquity + jewish_war) - sentences 1.xlsx"
)

BEST_MODEL_PATH = os.path.join(BASE_DIR, "exports", "exp3_model_state.pt")
STYLE_SCALER_PATH = os.path.join(BASE_DIR, "exports", "style_scaler_EXP1.pkl")

# אם את רוצה שלא תקלידי ידנית thresholds:
CALIB_JSON_PATH = os.path.join(BASE_DIR, "exports", "exp3_thresholds.json")

import json
with open(CALIB_JSON_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)
TAU_EN = float(cfg["thr_en"])
TAU_GR = float(cfg["thr_gr"])
MAX_LEN = int(cfg.get("MAX_LEN", 192))
print("✅ loaded thresholds:", TAU_EN, TAU_GR, "| MAX_LEN:", MAX_LEN)

BATCH_SIZE = 2
FORCE_CPU = False

# Output
OUT_DIR = os.path.join(BASE_DIR, "outputs")
OUT_PREFIX = os.path.join(OUT_DIR, "TaggedNico_Validation_EXP3")
os.makedirs(OUT_DIR, exist_ok=True)

print("✅ BASE_DIR:", BASE_DIR)
print("✅ INPUT_XLSX:", INPUT_XLSX)
print("✅ EXP3 MODEL:", BEST_MODEL_PATH)
print("✅ STYLE SCALER:", STYLE_SCALER_PATH)
print("✅ OUT_DIR:", OUT_DIR)
print("✅ TAU_EN/TAU_GR:", TAU_EN, TAU_GR)


✅ loaded thresholds: 0.32499999999999996 0.9049999999999999 | MAX_LEN: 192
✅ BASE_DIR: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית
✅ INPUT_XLSX: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\data\inputs\Tagged_Nico_TAG 5 vs Tag 4 (update model 6+5) vs TAG 0  (antiquity + jewish_war) - sentences 1.xlsx
✅ EXP3 MODEL: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\exports\exp3_model_state.pt
✅ STYLE SCALER: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\exports\style_scaler_EXP1.pkl
✅ OUT_DIR: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\outputs
✅ TAU_EN/TAU_GR: 0.32499999999999996 0.9049999999999999


In [2]:
# ============================
# 2) Imports + device
# ============================
import re, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import HashingVectorizer
import joblib

DEVICE = "cpu" if FORCE_CPU else ("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


DEVICE: cuda


In [3]:
# ============================
# 3) Load Excel (with lang inference + normalize pos/lemma)
# ============================
import pandas as pd
import re

df = pd.read_excel(INPUT_XLSX)

# Normalize Text column
if "Text" not in df.columns:
    for cand in ["text", "Sentence", "sentence"]:
        if cand in df.columns:
            df["Text"] = df[cand]
            break

# Normalize POS/Lemma column names for later cells
if "pos" not in df.columns:
    if "Pos" in df.columns:
        df["pos"] = df["Pos"]
if "lemma" not in df.columns:
    if "Lema" in df.columns:
        df["lemma"] = df["Lema"]

print("✅ Loaded rows:", len(df))
print("Columns:", list(df.columns))

# Validate Text exists
if "Text" not in df.columns:
    raise ValueError(f"Missing required column: 'Text'. Available: {list(df.columns)}")

# --- Build/normalize lang column ---
if "lang" not in df.columns:
    # Heuristic: detect Greek characters in the text
    greek_re = re.compile(r"[\u0370-\u03FF\u1F00-\u1FFF]")  # Greek + Greek Extended

    def infer_lang(s):
        s = "" if pd.isna(s) else str(s)
        return "gr" if greek_re.search(s) else "en"

    df["lang"] = df["Text"].apply(infer_lang)

df["lang"] = df["lang"].astype(str).str.lower()

print("✅ lang value counts:\n", df["lang"].value_counts())
print("✅ pos/lemma present:", ("pos" in df.columns), ("lemma" in df.columns))

# Optional: sanity check POS/Lemma exist (EXP3 needs them)
missing_pl = [c for c in ["pos", "lemma"] if c not in df.columns]
if missing_pl:
    print("⚠️ Missing POS/Lemma columns for EXP3:", missing_pl,
          "\n   Expected either pos/lemma or Pos/Lema in the Excel.")

✅ Loaded rows: 4072
Columns: ['Text', 'Pos', 'Lema', 'Predictions.-.Model.Tag.5.(c6./.c2)', 'Confidence_Class_0.-.Model.Tag.5.(c6./.c2)', 'Confidence_Class_1.-.Model.Tag.5.(c6./.c2)', 'Predictions.-.Model.Tag.5.(c5./.c1)', 'Confidence_Class_0.-.Model.Tag.5.(c5./.c1)', 'Confidence_Class_1.-.Model.Tag.5.(c5./.c1)', 'Predictions.-.Model.Tag.4.(c6./.c2)', 'Confidence_Class_0.-.Model.Tag.4.(c6./.c2)', 'Confidence_Class_1.-.Model.Tag.4.(c6./.c2)', 'Predictions.-.Model.Tag.4.(c5./.c1)', 'Confidence_Class_0.-.Model.Tag.4.(c5./.c1)', 'Confidence_Class_1.-.Model.Tag.4.(c5./.c1)', 'classifier_0_1_RandomForestClassifier', 'prob_classifier_0_1_RandomForestClassifier_0', 'prob_classifier_0_1_RandomForestClassifier_1', 'classifier_0_2_MultinomialNB', 'prob_classifier_0_2_MultinomialNB_0', 'prob_classifier_0_2_MultinomialNB_1', 'classifier_0_or_else_MultinomialNB', 'prob_classifier_0_or_else_MultinomialNB_0', 'prob_classifier_0_or_else_MultinomialNB_1', 'classifier_0_1_2_MultinomialNB', 'prob_classifi

In [4]:
# ============================
# 4) Feature builders (style + char + POS/Lemma hashing)
# ============================

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return summed / denom

# --- Style features (match Notebook B style builder as closely as possible) ---
WS_RE = re.compile(r"\s+")

def _count_diacritics(s: str) -> int:
    # combining marks
    return sum(1 for ch in s if unicodedata.combining(ch) != 0)

def style_features(s: str):
    s = "" if s is None else str(s)
    n = max(len(s), 1)
    words = [w for w in WS_RE.split(s.strip()) if w]
    wlen = [len(w) for w in words]

    letters = sum(ch.isalpha() for ch in s)
    digits  = sum(ch.isdigit() for ch in s)
    spaces  = s.count(" ")
    punct   = sum(ch in ".,;:!?" for ch in s)
    diac    = _count_diacritics(s)

    return [
        len(words),
        float(np.mean(wlen)) if wlen else 0.0,
        float(np.std(wlen)) if wlen else 0.0,
        letters / n,
        digits / n,
        spaces / n,
        punct / n,
        diac / n,
        len(set(s)) / n,
    ]

def build_style_matrix_from_df(df_in):
    texts = df_in["Text"].fillna("").astype(str).tolist()
    feats = []
    for s in texts:
        n = max(len(s), 1)
        words = [w for w in WS_RE.split(s.strip()) if w]
        wlen = [len(w) for w in words]
        feats.append([
            len(words),                                      # 1 word_count
            float(np.mean(wlen)) if wlen else 0.0,           # 2 avg_word_len
            float(np.std(wlen)) if wlen else 0.0,            # 3 std_word_len
            s.count(" ") / n,                                # 4 space_ratio
            sum(c.isdigit() for c in s) / n,                 # 5 digit_ratio
            sum(c in ".,;:!?" for c in s) / n,               # 6 punct_ratio
            len(set(s)) / n,                                 # 7 unique_char_ratio
        ])
    return np.array(feats, dtype=np.float32)

print("✅ style_dim =", build_style_matrix_from_df(df).shape[1])

# --- Char hashing ---
CHAR_DIM = 4096
char_vectorizer = HashingVectorizer(
    n_features=CHAR_DIM,
    analyzer="char",
    ngram_range=(3, 5),
    alternate_sign=False,
    norm=None,
)

# --- POS/Lemma hashing ---
POS_DIM = 2048
LEM_DIM = 2048
pos_vec = HashingVectorizer(n_features=POS_DIM, analyzer="word", ngram_range=(1,3), alternate_sign=False, norm=None)
lem_vec = HashingVectorizer(n_features=LEM_DIM, analyzer="word", ngram_range=(1,2), alternate_sign=False, norm=None)

def get_poslemma_cols(df_in):
    pos_col = "pos" if "pos" in df_in.columns else ("Pos" if "Pos" in df_in.columns else None)
    lem_col = "lemma" if "lemma" in df_in.columns else ("Lema" if "Lema" in df_in.columns else None)
    return pos_col, lem_col

# --- Load scaler ---
style_scaler = None
if os.path.exists(STYLE_SCALER_PATH):
    style_scaler = joblib.load(STYLE_SCALER_PATH)
    print("✅ loaded style_scaler:", STYLE_SCALER_PATH)
else:
    print("⚠️ STYLE_SCALER_PATH not found, continuing without scaling:", STYLE_SCALER_PATH)

print("✅ style_dim:", build_style_matrix_from_df(df).shape[1])


✅ style_dim = 7
✅ loaded style_scaler: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\exports\style_scaler_EXP1.pkl
✅ style_dim: 7


In [5]:
# ============================
# 5) Load EXP3 model (style + char + POS + lemma)
# ============================

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grl(x, alpha=1.0):
    return GradReverse.apply(x, alpha)

class XLMR_StyleCharPosLemma_Adv(nn.Module):
    def __init__(self, model_name, style_dim, char_dim, pos_dim, lem_dim,
                 style_h=32, char_h=128, pos_h=64, lem_h=64, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)
        h = self.encoder.config.hidden_size

        self.style_mlp = nn.Sequential(nn.Linear(style_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, style_h))
        self.char_mlp  = nn.Sequential(nn.Linear(char_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, char_h))
        self.pos_mlp   = nn.Sequential(nn.Linear(pos_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, pos_h))
        self.lem_mlp   = nn.Sequential(nn.Linear(lem_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, lem_h))

        self.g_style = nn.Parameter(torch.tensor(-1.0))
        self.g_char  = nn.Parameter(torch.tensor(-1.0))
        self.g_pos   = nn.Parameter(torch.tensor(-1.0))
        self.g_lem   = nn.Parameter(torch.tensor(-1.0))

        fused_dim = h + style_h + char_h + pos_h + lem_h
        self.ln = nn.LayerNorm(fused_dim)
        self.drop = nn.Dropout(dropout)

        self.rewrite_head = nn.Linear(fused_dim, 2)
        self.lang_head    = nn.Linear(fused_dim, 2)

    def forward(self, tok, style_vec, char_vec, pos_vec, lem_vec, grl_alpha=1.0):
        out = self.encoder(**tok, return_dict=True)
        h_text = mean_pool(out.last_hidden_state, tok["attention_mask"])

        h_style = self.style_mlp(style_vec)
        h_char  = self.char_mlp(char_vec)
        h_pos   = self.pos_mlp(pos_vec)
        h_lem   = self.lem_mlp(lem_vec)

        gs = torch.sigmoid(self.g_style)
        gc = torch.sigmoid(self.g_char)
        gp = torch.sigmoid(self.g_pos)
        gl = torch.sigmoid(self.g_lem)

        fused = torch.cat([h_text, gs*h_style, gc*h_char, gp*h_pos, gl*h_lem], dim=-1)
        fused = self.drop(self.ln(fused))

        logits_y    = self.rewrite_head(fused)
        logits_lang = self.lang_head(grl(fused, grl_alpha))
        return logits_y, logits_lang

# tokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base", local_files_only=True)

STYLE_DIM = build_style_matrix_from_df(df).shape[1]
model = XLMR_StyleCharPosLemma_Adv("xlm-roberta-base", STYLE_DIM, CHAR_DIM, POS_DIM, LEM_DIM).to(DEVICE)

sd = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(sd, strict=True)
model.eval()

print("✅ Loaded EXP3 model state:", BEST_MODEL_PATH)


✅ Loaded EXP3 model state: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\exports\exp3_model_state.pt


In [6]:
# ============================
# 6) Inference + save
# ============================

@torch.no_grad()
def predict_probs(df_in):
    texts = df_in["Text"].fillna("").astype(str).tolist()

    # style
    Xs = build_style_matrix_from_df(df_in)
    if style_scaler is not None:
        Xs = style_scaler.transform(Xs).astype(np.float32)
    else:
        Xs = Xs.astype(np.float32)

    # char sparse
    Xc = char_vectorizer.transform(texts).astype(np.float32)

    # pos/lemma
    pos_col, lem_col = get_poslemma_cols(df_in)
    if pos_col is None or lem_col is None:
        raise ValueError(
            "POS/Lemma columns not found in Excel. Need 'pos'+'lemma' (or 'Pos'+'Lema').\n"
            f"Available columns: {list(df_in.columns)}"
        )
    X_pos = pos_vec.transform(df_in[pos_col].fillna("").astype(str)).astype(np.float32)
    X_lem = lem_vec.transform(df_in[lem_col].fillna("").astype(str)).astype(np.float32)

    probs = np.zeros(len(df_in), dtype=np.float32)

    for i in range(0, len(df_in), BATCH_SIZE):
        batch_texts = texts[i:i+BATCH_SIZE]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        style = torch.tensor(Xs[i:i+BATCH_SIZE], dtype=torch.float32, device=DEVICE)
        char  = torch.tensor(Xc[i:i+BATCH_SIZE].toarray().astype(np.float32), dtype=torch.float32, device=DEVICE)
        pos   = torch.tensor(X_pos[i:i+BATCH_SIZE].toarray().astype(np.float32), dtype=torch.float32, device=DEVICE)
        lem   = torch.tensor(X_lem[i:i+BATCH_SIZE].toarray().astype(np.float32), dtype=torch.float32, device=DEVICE)

        logits_y, _ = model(tok, style, char, pos, lem, grl_alpha=0.0)
        probs[i:i+BATCH_SIZE] = torch.softmax(logits_y, dim=-1)[:, 1].detach().cpu().numpy()

    return probs

def apply_thresholds(df_in, p):
    langs = df_in["lang"].astype(str).str.lower().values
    tau = np.where(langs == "gr", TAU_GR, TAU_EN)
    return (p >= tau).astype(int)

p = predict_probs(df)
yhat = apply_thresholds(df, p)

df["Prob Rewrite"] = p

# output frame (keep only key columns that exist)
cols = []
for c in ["Book", "Text", "Nico_Matched", "Prob Rewrite"]:
    if c in df.columns:
        cols.append(c)

# ============================
# 6) Inference + save
# ============================

@torch.no_grad()
def predict_probs(df_in):
    texts = df_in["Text"].fillna("").astype(str).tolist()

    # style
    Xs = build_style_matrix_from_df(df_in)
    if style_scaler is not None:
        Xs = style_scaler.transform(Xs).astype(np.float32)
    else:
        Xs = Xs.astype(np.float32)

    # char sparse
    Xc = char_vectorizer.transform(texts).astype(np.float32)

    # pos/lemma
    pos_col, lem_col = get_poslemma_cols(df_in)
    if pos_col is None or lem_col is None:
        raise ValueError(
            "POS/Lemma columns not found in Excel. Need 'pos'+'lemma' (or 'Pos'+'Lema').\n"
            f"Available columns: {list(df_in.columns)}"
        )
    X_pos = pos_vec.transform(df_in[pos_col].fillna("").astype(str)).astype(np.float32)
    X_lem = lem_vec.transform(df_in[lem_col].fillna("").astype(str)).astype(np.float32)

    probs = np.zeros(len(df_in), dtype=np.float32)

    for i in range(0, len(df_in), BATCH_SIZE):
        batch_texts = texts[i:i+BATCH_SIZE]
        tok = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        style = torch.tensor(Xs[i:i+BATCH_SIZE], dtype=torch.float32, device=DEVICE)
        char  = torch.tensor(Xc[i:i+BATCH_SIZE].toarray().astype(np.float32), dtype=torch.float32, device=DEVICE)
        pos   = torch.tensor(X_pos[i:i+BATCH_SIZE].toarray().astype(np.float32), dtype=torch.float32, device=DEVICE)
        lem   = torch.tensor(X_lem[i:i+BATCH_SIZE].toarray().astype(np.float32), dtype=torch.float32, device=DEVICE)

        logits_y, _ = model(tok, style, char, pos, lem, grl_alpha=0.0)
        probs[i:i+BATCH_SIZE] = torch.softmax(logits_y, dim=-1)[:, 1].detach().cpu().numpy()

    return probs


def apply_thresholds(df_in, p):
    langs = df_in["lang"].astype(str).str.lower().values
    tau = np.where(langs == "gr", TAU_GR, TAU_EN)
    return (p >= tau).astype(int)


# ---- predict ----
p = predict_probs(df)
yhat = apply_thresholds(df, p)

# keep in full df too
df["EXP3 Prob Rewrite"] = p
df["EXP3 Pred Label"] = yhat
df["EXP3 Pred Category"] = np.where(yhat == 1, "Rewrite", "Original")

# stable row id for row-wise comparison to source file
df["Source Row"] = np.arange(len(df))

# base columns that are useful for comparison
base_cols = []
for c in [
    "Source Row",
    "Book",
    "Book.Title",
    "Chapter.Start",
    "Chapter.End",
    "Paragraph.Start",
    "Paragraph.End",
    "Sentence.Number",
    "Text",
    "Category",
    "Nico_Matched"
]:
    if c in df.columns:
        base_cols.append(c)

# keep original model prediction / confidence columns from source excel
original_pred_cols = [
    c for c in df.columns
    if (
        c.startswith("Predictions")
        or c.startswith("Confidence")
        or c.startswith("classifier_")
        or c.startswith("prob_classifier_")
    )
]

# avoid duplicates while preserving order
cols = []
seen = set()
for c in base_cols + original_pred_cols:
    if c not in seen and c in df.columns:
        cols.append(c)
        seen.add(c)

df_pred = df[cols].copy()

# append current EXP3 outputs
df_pred["EXP3 Prob Rewrite"] = p
df_pred["EXP3 Pred Label"] = yhat
df_pred["EXP3 Pred Category"] = np.where(yhat == 1, "Rewrite", "Original")

out_xlsx = f"{OUT_PREFIX}__PREDICTIONS.xlsx"
df_pred.to_excel(out_xlsx, index=False)

print("✅ Saved:", out_xlsx)
print("Rows:", len(df_pred))
print("Original prediction/confidence columns kept:", len(original_pred_cols))

df_pred.head(5)


✅ Saved: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\outputs\TaggedNico_Validation_EXP3__PREDICTIONS.xlsx
Rows: 4072
Original prediction/confidence columns kept: 25


,Source Row,Book.Title,Chapter.Start,Chapter.End,Paragraph.Start,Paragraph.End,Sentence.Number,Text,Category,Nico_Matched,...,classifier_0_or_else_MultinomialNB,prob_classifier_0_or_else_MultinomialNB_0,prob_classifier_0_or_else_MultinomialNB_1,classifier_0_1_2_MultinomialNB,prob_classifier_0_1_2_MultinomialNB_0,prob_classifier_0_1_2_MultinomialNB_1,prob_classifier_0_1_2_MultinomialNB_2,EXP3 Prob Rewrite,EXP3 Pred Label,EXP3 Pred Category
0,0,antiquity1_1,1.0,1.0,1.0,1.0,1.0,τοῖς τὰς ἱστορίας συγγράφειν βουλομένοις οὐ μί...,NaN,NaN,...,0,0.996100,0.003900,2,0.346931,0.285317,0.367752,0.034138,0,Original
1,1,antiquity1_1,1.0,1.0,2.0,3.0,2.0,τινὲς μὲν γὰρ ἐπιδεικνύμενοι λόγων δεινότητα κ...,NaN,NaN,...,0,0.999165,0.000835,2,0.343823,0.286209,0.369969,0.167622,0,Original
2,2,antiquity1_1,1.0,2.0,4.0,5.0,3.0,τούτων δὴ τῶν προειρημένων αἰτιῶν αἱ τελευταῖα...,NaN,NaN,...,0,0.999721,0.000279,2,0.346840,0.285171,0.367989,0.054096,0,Original
3,3,antiquity1_1,2.0,2.0,6.0,6.0,4.0,ἤδη μὲν οὖν καὶ πρότερον διενοήθην ὅτε τὸν πόλ...,NaN,NaN,...,0,0.997368,0.002632,2,0.332848,0.303669,0.363484,0.311577,0,Original
4,4,antiquity1_1,2.0,2.0,7.0,7.0,5.0,ἀλλʼ ἐπειδὴ μείζων ἦν ἡ τοῦδε τοῦ λόγου περιβο...,NaN,NaN,...,0,0.943817,0.056183,2,0.321854,0.329847,0.348300,0.092952,0,Original


In [7]:
# ============================
# Greek-only EXP3 analysis vs Nico proxy labels
# Correct mapping:
#   Nico_Matched = 1     => Original (0)
#   Nico_Matched = empty => Rewrite  (1)
#   Nico_Matched in {2,3} => ignore
# ============================
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score
import os

pred_path = f"{OUT_PREFIX}__PREDICTIONS.xlsx"

print("Reading:", pred_path)
assert os.path.exists(pred_path), f"File not found: {pred_path}"

dfp = pd.read_excel(pred_path)
print("✅ loaded:", dfp.shape)
print("Columns:\n", list(dfp.columns))

# ----------------------------
# Resolve column names robustly
# ----------------------------
prob_col = None
for c in ["EXP3 Prob Rewrite", "Prob Rewrite"]:
    if c in dfp.columns:
        prob_col = c
        break

pred_col = None
for c in ["EXP3 Pred Label", "Pred Label"]:
    if c in dfp.columns:
        pred_col = c
        break

pred_cat_col = None
for c in ["EXP3 Pred Category", "Pred Category"]:
    if c in dfp.columns:
        pred_cat_col = c
        break

if prob_col is None:
    raise KeyError(
        "Could not find probability column. Expected one of: "
        "['EXP3 Prob Rewrite', 'Prob Rewrite'].\n"
        f"Available columns: {list(dfp.columns)}"
    )

print(f"Using probability column: {prob_col}")
print(f"Using predicted-label column: {pred_col if pred_col is not None else 'None (will derive from threshold)'}")
print(f"Using predicted-category column: {pred_cat_col if pred_cat_col is not None else 'None'}")

# ----------------------------
# Build corrected proxy labels from Nico_Matched
# Correct mapping:
#   1     -> Original (0)
#   empty -> Rewrite  (1)
#   2,3   -> ignore
# ----------------------------
nico = dfp["Nico_Matched"] if "Nico_Matched" in dfp.columns else pd.Series([np.nan] * len(dfp))
nico_str = nico.fillna("").astype(str).str.strip()

is_one   = nico_str.isin(["1", "1.0"])
is_empty = nico.isna() | nico_str.eq("")
is_23    = nico_str.isin(["2", "2.0", "3", "3.0"])

# keep only rows relevant for proxy comparison
use_mask = is_one | is_empty
use = dfp.loc[use_mask].copy()

# corrected label mapping
# 1 => Original (0), empty => Rewrite (1)
use["y_proxy"] = np.where(is_one[use_mask], 0, 1).astype(int)

# ----------------------------
# Predictions / probabilities
# ----------------------------
use["p"] = pd.to_numeric(use[prob_col], errors="coerce")

if pred_col is not None:
    use["pred_file"] = pd.to_numeric(use[pred_col], errors="coerce")
else:
    use["pred_file"] = (use["p"] >= TAU_GR).astype(int)

before_drop = len(use)
use = use[use["p"].notna() & use["pred_file"].notna()].copy()
use["pred_file"] = use["pred_file"].astype(int)

print("Rows total:", len(dfp))
print("Rows ignored because Nico_Matched in {2,3,other}:", len(dfp) - int(use_mask.sum()))
print("Rows used before numeric cleanup:", before_drop)
print("Rows used after numeric cleanup:", len(use))
print("Proxy class counts:\n", use["y_proxy"].value_counts().sort_index().to_string())

# ----------------------------
# 1) Basic quality without changing threshold
# ----------------------------
y_true = use["y_proxy"].values
y_pred = use["pred_file"].values
p = use["p"].values

print("\n=== Current (as saved) confusion/report ===")
print("Confusion (rows=true 0/1, cols=pred 0/1):\n", confusion_matrix(y_true, y_pred, labels=[0, 1]))
print(classification_report(y_true, y_pred, digits=3))

if len(np.unique(y_true)) == 2:
    print("ROC-AUC:", roc_auc_score(y_true, p))
    print("PR-AUC :", average_precision_score(y_true, p))

# ----------------------------
# 2) Distribution separation
# ----------------------------
print("\n=== Probability separation (proxy 0 vs 1) ===")
for lab in [0, 1]:
    s = use.loc[use["y_proxy"] == lab, "p"]
    if len(s) == 0:
        print(f"y_proxy={lab} | n=0")
    else:
        print(
            f"y_proxy={lab} | n={len(s)} | "
            f"min={s.min():.3f} p25={s.quantile(0.25):.3f} median={s.median():.3f} "
            f"p75={s.quantile(0.75):.3f} max={s.max():.3f} mean={s.mean():.3f}"
        )

# ----------------------------
# 3) Threshold sweep
# FN slightly more important than FP
# Here positive class = Rewrite (1)
# ----------------------------
FN_W = 1.2
FP_W = 1.0

thresholds = np.linspace(0.01, 0.99, 199)
rows = []

for t in thresholds:
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    f2   = 5 * prec * rec / (4 * prec + rec) if (4 * prec + rec) else 0.0
    cost = FN_W * fn + FP_W * fp
    rows.append((t, prec, rec, f1, f2, fp, fn, tp, tn, cost))

sweep = pd.DataFrame(rows, columns=["thr", "prec", "rec", "f1", "f2", "fp", "fn", "tp", "tn", "cost"])
best_cost = sweep.sort_values(["cost", "f2"], ascending=[True, False]).head(1)
best_f1   = sweep.sort_values("f1", ascending=False).head(1)
best_f2   = sweep.sort_values("f2", ascending=False).head(1)

print("\n=== Best thresholds (proxy) ===")
print("Best by COST (1.2*FN + 1.0*FP):")
print(best_cost.to_string(index=False))
print("\nBest by F1:")
print(best_f1.to_string(index=False))
print("\nBest by F2:")
print(best_f2.to_string(index=False))

# ----------------------------
# 4) Review buckets
# positive class is Rewrite (1)
# FP: proxy=0 but p high
# FN: proxy=1 but p low
# ----------------------------
fp_examples = use[use["y_proxy"] == 0].sort_values("p", ascending=False).head(30).copy()
fp_examples["error_type"] = "FP_candidate (proxy=Original, high rewrite p)"

fn_examples = use[use["y_proxy"] == 1].sort_values("p", ascending=True).head(30).copy()
fn_examples["error_type"] = "FN_candidate (proxy=Rewrite, low rewrite p)"

review = pd.concat([fp_examples, fn_examples], ignore_index=True)

review_cols = ["error_type"]
for c in [
    "Source Row",
    "Text",
    "Nico_Matched",
    "y_proxy",
    prob_col,
    pred_col,
    pred_cat_col
]:
    if c is not None and c in review.columns and c not in review_cols:
        review_cols.append(c)

review["p"] = review[prob_col]
if pred_col is not None and pred_col in review.columns:
    review["pred_file"] = review[pred_col]
if pred_cat_col is not None and pred_cat_col in review.columns:
    review["pred_category_file"] = review[pred_cat_col]

for c in ["p", "pred_file", "pred_category_file"]:
    if c in review.columns and c not in review_cols:
        review_cols.append(c)

review = review[review_cols].copy()

# ----------------------------
# 5) Save diagnostics workbook
# ----------------------------
out_path = "TaggedNico_Validation_EXP3__GREEK_DIAGNOSTICS.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    use.to_excel(w, index=False, sheet_name="used_rows_proxy")
    sweep.to_excel(w, index=False, sheet_name="threshold_sweep")
    best_cost.to_excel(w, index=False, sheet_name="best_cost")
    best_f1.to_excel(w, index=False, sheet_name="best_f1")
    best_f2.to_excel(w, index=False, sheet_name="best_f2")
    review.to_excel(w, index=False, sheet_name="review_60_examples")

print("\n✅ Saved diagnostics:", out_path)

Reading: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\outputs\TaggedNico_Validation_EXP3__PREDICTIONS.xlsx
✅ loaded: (4072, 38)
Columns:
 ['Source Row', 'Book.Title', 'Chapter.Start', 'Chapter.End', 'Paragraph.Start', 'Paragraph.End', 'Sentence.Number', 'Text', 'Category', 'Nico_Matched', 'Predictions.-.Model.Tag.5.(c6./.c2)', 'Confidence_Class_0.-.Model.Tag.5.(c6./.c2)', 'Confidence_Class_1.-.Model.Tag.5.(c6./.c2)', 'Predictions.-.Model.Tag.5.(c5./.c1)', 'Confidence_Class_0.-.Model.Tag.5.(c5./.c1)', 'Confidence_Class_1.-.Model.Tag.5.(c5./.c1)', 'Predictions.-.Model.Tag.4.(c6./.c2)', 'Confidence_Class_0.-.Model.Tag.4.(c6./.c2)', 'Confidence_Class_1.-.Model.Tag.4.(c6./.c2)', 'Predictions.-.Model.Tag.4.(c5./.c1)', 'Confidence_Class_0.-.Model.Tag.4.(c5./.c1)', 'Confidence_Class_1.-.Model.Tag.4.(c5./.c1)', 'classifier_0_1_RandomForestClassifier', 'prob_classifier_0_1_RandomForestClassifier_0', 'prob_classifier_0_1_RandomForestClassifier_1', 'classifier_0_2_MultinomialNB', 'prob_classifi

In [8]:
# ============================
# Compare EXP3 vs old model:
# Predictions.-.Model.Tag.5.(c6./.c2)
# ============================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

pred_path = f"{OUT_PREFIX}__PREDICTIONS.xlsx"

print("Reading:", pred_path)
assert os.path.exists(pred_path), f"File not found: {pred_path}"

dfc = pd.read_excel(pred_path)
print("✅ loaded:", dfc.shape)

# -------------------------------------------------
# Resolve columns
# -------------------------------------------------
exp3_pred_col = None
for c in ["EXP3 Pred Label", "Pred Label"]:
    if c in dfc.columns:
        exp3_pred_col = c
        break

exp3_prob_col = None
for c in ["EXP3 Prob Rewrite", "Prob Rewrite"]:
    if c in dfc.columns:
        exp3_prob_col = c
        break

old_pred_col = "Predictions.-.Model.Tag.5.(c6./.c2)"
old_prob0_col = "Confidence_Class_0.-.Model.Tag.5.(c6./.c2)"
old_prob1_col = "Confidence_Class_1.-.Model.Tag.5.(c6./.c2)"

if exp3_pred_col is None:
    raise KeyError(
        "Could not find EXP3 prediction column. Expected one of "
        "['EXP3 Pred Label', 'Pred Label'].\n"
        f"Available columns: {list(dfc.columns)}"
    )

if old_pred_col not in dfc.columns:
    raise KeyError(
        f"Could not find old prediction column: {old_pred_col}\n"
        f"Available columns: {list(dfc.columns)}"
    )

print("Using EXP3 pred column:", exp3_pred_col)
print("Using EXP3 prob column:", exp3_prob_col if exp3_prob_col is not None else "None")
print("Using old pred column:", old_pred_col)

# -------------------------------------------------
# Prepare comparison frame
# -------------------------------------------------
cmp = dfc.copy()

cmp["exp3_pred"] = pd.to_numeric(cmp[exp3_pred_col], errors="coerce")
cmp["old_pred"] = pd.to_numeric(cmp[old_pred_col], errors="coerce")

if exp3_prob_col is not None:
    cmp["exp3_prob"] = pd.to_numeric(cmp[exp3_prob_col], errors="coerce")

if old_prob0_col in cmp.columns:
    cmp["old_prob_0"] = pd.to_numeric(cmp[old_prob0_col], errors="coerce")
if old_prob1_col in cmp.columns:
    cmp["old_prob_1"] = pd.to_numeric(cmp[old_prob1_col], errors="coerce")

before = len(cmp)
cmp = cmp[cmp["exp3_pred"].notna() & cmp["old_pred"].notna()].copy()
cmp["exp3_pred"] = cmp["exp3_pred"].astype(int)
cmp["old_pred"] = cmp["old_pred"].astype(int)

print("Rows before cleanup:", before)
print("Rows after cleanup :", len(cmp))

# -------------------------------------------------
# Agreement stats
# -------------------------------------------------
cm = confusion_matrix(cmp["old_pred"], cmp["exp3_pred"], labels=[0, 1])
agreement = (cmp["old_pred"] == cmp["exp3_pred"]).mean()

print("\n=== EXP3 vs old model ===")
print("Confusion (rows=old model 0/1, cols=EXP3 0/1):\n", cm)
print(f"Agreement: {agreement:.4%}")

print("\n=== Classification report (old model as reference) ===")
print(classification_report(cmp["old_pred"], cmp["exp3_pred"], digits=3))

# -------------------------------------------------
# Disagreement buckets
# -------------------------------------------------
cmp["agreement"] = np.where(cmp["old_pred"] == cmp["exp3_pred"], "agree", "disagree")

# old=1, exp3=0
d10 = cmp[(cmp["old_pred"] == 1) & (cmp["exp3_pred"] == 0)].copy()
d10["disagreement_type"] = "old=1_EXP3=0"

# old=0, exp3=1
d01 = cmp[(cmp["old_pred"] == 0) & (cmp["exp3_pred"] == 1)].copy()
d01["disagreement_type"] = "old=0_EXP3=1"

# Sort disagreements by confidence if available
if "exp3_prob" in d10.columns:
    d10 = d10.sort_values("exp3_prob", ascending=True)
if "exp3_prob" in d01.columns:
    d01 = d01.sort_values("exp3_prob", ascending=False)

disagreements = pd.concat([d10, d01], ignore_index=True)

# smaller manual-review sample
review_parts = []

if len(d10) > 0:
    review_parts.append(d10.head(30).copy())
if len(d01) > 0:
    review_parts.append(d01.head(30).copy())

review = pd.concat(review_parts, ignore_index=True) if review_parts else pd.DataFrame()

# -------------------------------------------------
# Friendly columns for export
# -------------------------------------------------
export_cols = []
for c in [
    "Source Row",
    "Book.Title",
    "Chapter.Start",
    "Chapter.End",
    "Paragraph.Start",
    "Paragraph.End",
    "Sentence.Number",
    "Text",
    "Category",
    "Nico_Matched",
    old_pred_col,
    old_prob0_col,
    old_prob1_col,
    exp3_prob_col,
    exp3_pred_col,
    "exp3_pred",
    "old_pred",
    "agreement",
    "disagreement_type",
]:
    if c is not None and c in cmp.columns and c not in export_cols:
        export_cols.append(c)

cmp_export = cmp[export_cols].copy()

review_cols = []
for c in [
    "disagreement_type",
    "Source Row",
    "Book.Title",
    "Chapter.Start",
    "Chapter.End",
    "Paragraph.Start",
    "Paragraph.End",
    "Sentence.Number",
    "Text",
    "Category",
    "Nico_Matched",
    old_pred_col,
    old_prob0_col,
    old_prob1_col,
    exp3_prob_col,
    exp3_pred_col,
    "old_pred",
    "exp3_pred",
]:
    if c is not None and c in review.columns and c not in review_cols:
        review_cols.append(c)

if not review.empty:
    review = review[review_cols].copy()

# -------------------------------------------------
# Summary table
# -------------------------------------------------
summary = pd.DataFrame([
    {"metric": "rows_compared", "value": len(cmp)},
    {"metric": "agreement_rate", "value": float(agreement)},
    {"metric": "agree_count", "value": int((cmp["agreement"] == "agree").sum())},
    {"metric": "disagree_count", "value": int((cmp["agreement"] == "disagree").sum())},
    {"metric": "old_1_exp3_0", "value": int(((cmp["old_pred"] == 1) & (cmp["exp3_pred"] == 0)).sum())},
    {"metric": "old_0_exp3_1", "value": int(((cmp["old_pred"] == 0) & (cmp["exp3_pred"] == 1)).sum())},
])

cm_df = pd.DataFrame(
    cm,
    index=["old_0", "old_1"],
    columns=["exp3_0", "exp3_1"]
)

# -------------------------------------------------
# Save workbook
# -------------------------------------------------
out_path = "TaggedNico_EXP3_vs_OldModel_Tag5_c6c2.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary.to_excel(w, index=False, sheet_name="summary")
    cm_df.to_excel(w, sheet_name="confusion_matrix")
    cmp_export.to_excel(w, index=False, sheet_name="all_rows_comparison")
    disagreements.to_excel(w, index=False, sheet_name="all_disagreements")
    if not review.empty:
        review.to_excel(w, index=False, sheet_name="review_60_examples")

print("\n✅ Saved comparison workbook:", out_path)

Reading: C:\\Users\\Asoulin_Sapir\\Desktop\\עבודה עדכנית\outputs\TaggedNico_Validation_EXP3__PREDICTIONS.xlsx
✅ loaded: (4072, 38)
Using EXP3 pred column: EXP3 Pred Label
Using EXP3 prob column: EXP3 Prob Rewrite
Using old pred column: Predictions.-.Model.Tag.5.(c6./.c2)
Rows before cleanup: 4072
Rows after cleanup : 4072

=== EXP3 vs old model ===
Confusion (rows=old model 0/1, cols=EXP3 0/1):
 [[3140  566]
 [ 233  133]]
Agreement: 80.3782%

=== Classification report (old model as reference) ===
              precision    recall  f1-score   support

           0      0.931     0.847     0.887      3706
           1      0.190     0.363     0.250       366

    accuracy                          0.804      4072
   macro avg      0.561     0.605     0.568      4072
weighted avg      0.864     0.804     0.830      4072


✅ Saved comparison workbook: TaggedNico_EXP3_vs_OldModel_Tag5_c6c2.xlsx
